In [34]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import numpy as np
import re

import os
from pathlib import Path
os.environ['USER_AGENT'] = 'myagent'

os.chdir("/YOUR PATH HERE")
cwd = Path.cwd()
notebook_path = os.getcwd()

from dotenv import load_dotenv
load_dotenv()  # <-- This is what tells Python to load variables from the .env file
api_key = os.getenv('openai_api_key')
api_org = os.getenv('openai_org')

# Confirm it's loaded (optional but helpful)
if not api_key:
    raise ValueError("OpenAI API key not found in environment variables!")

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from googleapiclient.http import MediaIoBaseDownload

import io
import json

import openai
from PyPDF2 import PdfReader  # or fitz (PyMuPDF), depending on your parser


In [4]:
#✅ Block 0: Google Street Cred
def get_credentials(service_account_file, scopes):
    """
    Load service account credentials from JSON file.

    Args:
        service_account_file (str): Path to service account JSON.
        scopes (list): List of OAuth2 scopes.

    Returns:
        google.oauth2.service_account.Credentials: Authorized credentials object.
    """
    try:
        credentials = service_account.Credentials.from_service_account_file(
            service_account_file,
            scopes=scopes
        )
        return credentials
    except Exception as e:
        print(f"❌ Failed to load credentials: {e}")
        raise

def get_drive_service(credentials, version='v3'):
    return build('drive', version, credentials=credentials)

def get_gspread_client(credentials):
    return gspread.authorize(credentials)

In [6]:
def load_roles_from_drive(drive_service, file_name, folder_id='XXX'):
    """
    Loads a roles JSON file from a specified Google Drive folder using folder ID.
    """
    # Search for the JSON file inside the folder
    file_query = f"name = '{file_name}' and '{folder_id}' in parents and trashed = false"
    file_results = drive_service.files().list(q=file_query, fields="files(id, name)").execute()
    files = file_results.get('files', [])
    
    if not files:
        raise Exception(f"File '{file_name}' not found in folder ID '{folder_id}'.")
    
    file_id = files[0]['id']
    
    # Download the JSON file content
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    
    done = False
    while not done:
        status, done = downloader.next_chunk()
    
    fh.seek(0)
    roles_data = json.load(fh)
    
    return roles_data


In [8]:
# File: creds.py (or whatever your module is)
SERVICE_ACCOUNT_FILE = 'YOUR-KEY-LOCATION-HERE.json'

SCOPES = [
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets'
]

credentials = get_credentials(SERVICE_ACCOUNT_FILE, SCOPES)

drive_service = get_drive_service(credentials)
gspread_client = get_gspread_client(credentials)

# Example: Load roles from Drive
# Replace with your actual folder ID for "Role Descriptions"
folder_id = 'YOUR-ROLES-FOLDER-ID-HERE',
file_name = 'roles_1.json'

roles_json = load_roles_from_drive(drive_service, file_name, folder_id=folder_id)
roles_list = roles_json['roles']

# Test output
for role in roles_list:
    print(role['title'], "-", role['context'])

ML Modeler for Auto Lending - Small subprime auto lender focused on rapid credit expansion in underserved markets. Limited regulatory oversight compared to large banks, facing high default rates and thin-file customers.
SVP for Risk Analytics - Consumer Lending - Large international bank subject to stringent regulations such as Basel III and CCAR. Emphasis on comprehensive risk management, regulatory reporting, and maintaining strong governance across consumer lending portfolios.
Fraud Data Scientist - Digital-first fintech focused on rapid onboarding with high exposure to synthetic identity fraud and account takeover risks.


In [9]:
def download_file_from_drive(drive_service, file_name, folder_id):
    """
    Downloads a PDF file from Google Drive and returns file content.
    """
    file_query = f"name = '{file_name}' and '{folder_id}' in parents and trashed = false"
    file_results = drive_service.files().list(q=file_query, fields="files(id, name)").execute()
    files = file_results.get('files', [])
    
    if not files:
        raise Exception(f"File '{file_name}' not found in folder ID '{folder_id}'.")
    
    file_id = files[0]['id']
    
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    
    done = False
    while not done:
        status, done = downloader.next_chunk()
    
    fh.seek(0)
    return fh

def extract_text_from_pdf(pdf_file_bytes):
    """
    Extracts text from a PDF file given as bytes.
    """
    reader = PdfReader(pdf_file_bytes)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

In [48]:

# Define API rates (per 1 million tokens)
api_rates = {
    'gpt-4.5-preview': 75,
    'gpt-4o': 2.5,
    'gpt-4o-audio-preview': 2.5,
    'gpt-4o-realtime-preview': 5,
    'gpt-4o-mini': 0.15,
    'gpt-4o-mini-audio-preview': 0.15,
    'gpt-4o-mini-realtime-preview': 0.6,
    'o1': 15,
    'o3-mini': 1.1,
    'o1-mini': 1.1
}


def summarize_document_for_roles(file_text, roles_list, model="gpt-4o", max_tokens=4000, temp=0.3):
    roles_prompt_section = ""
    for idx, role in enumerate(roles_list, start=1):
        roles_prompt_section += (
            f"{idx}. Role Title: {role['title']}\n"
            f"   Organization Context: {role['context']}\n"
            f"   Description: {role['description']}\n\n"
        )
    
    system_prompt = "You are an expert summarizer for different professional roles."

    user_prompt = f"""
TASK:
You will assess the relevance of the following document for multiple professional roles and provide tailored summaries accordingly.

1. For each role, assess how relevant the document is using this scale:
   - Very Relevant: The document contains significant and detailed information useful to this role.
   - Somewhat Relevant: The document contains limited information of interest to this role.
   - Not Relevant: The document is not applicable to this role.

2. For each role:
   - Provide a concise summary highlighting only the most critical points.
   - If the document is Very Relevant, explicitly start the summary with: "This document is highly relevant and should be read in full."
   - If Somewhat Relevant, no additional comments are necessary beyond the summary.
   - If Not Relevant, explicitly start the summary with: "This document does not contain relevant information and no further action is needed."

DOCUMENT:
\"\"\"
{file_text}
\"\"\"

ROLES:
{roles_prompt_section}

OUTPUT FORMAT:
For each role, provide:
Role Title: [Role Title]  
Relevance: [Very Relevant | Somewhat Relevant | Not Relevant]  
Summary:  
[Role-specific summary here or state 'Not Relevant']
"""

    # New API client call
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temp
    )

    summary_text = response.choices[0].message.content
    usage = response.usage

    input_tokens = usage.prompt_tokens
    output_tokens = usage.completion_tokens
    total_tokens = usage.total_tokens

    model_rate_per_million = api_rates.get(model, None)
    if model_rate_per_million is None:
        raise ValueError(f"Model '{model}' not found in api_rates dictionary.")

    cost_per_token = model_rate_per_million / 1_000_000
    total_cost = total_tokens * cost_per_token

    token_usage = {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "rate_per_million_usd": model_rate_per_million,
        "total_cost_usd": round(total_cost, 4)
    }

    return summary_text, token_usage

In [40]:
def run_live_test(drive_service, roles_folder_id, roles_doc_name, docs_folder_id, doc_filename, temp=0.3):
    # 1. Load roles JSON
    roles_json = load_roles_from_drive(drive_service, roles_doc_name, folder_id=roles_folder_id)
    roles_list = roles_json['roles']
    
    # 2. Download file.pdf
    pdf_file_bytes = download_file_from_drive(drive_service, doc_filename, folder_id=docs_folder_id)
    
    # 3. Extract text from PDF
    document_text = extract_text_from_pdf(pdf_file_bytes)
    
    if not document_text.strip():
        raise ValueError("PDF text extraction failed or returned empty content.")
    
    print("✅ Document text extracted.")
    
    # 4. Summarize
    summary_text, token_usage = summarize_document_for_roles(document_text, roles_list, model="gpt-4o-mini", temp=temp)
    
    print("📝 Summaries:\n")
    print(summary_text)

    print("\n📊 Token Usage & Cost:")
    for key, value in token_usage.items():
        print(f"{key}: {value}")

# Example credentials (fill these in with your config)
# credentials = get_credentials(service_account_file, scopes)
# drive_service = get_drive_service(credentials)

# --- Run It ---
# run_live_test(drive_service, roles_folder_id='XXX', docs_folder_id='YYY')

In [50]:
client = openai.OpenAI(
    api_key=api_key,
    organization=api_org)

run_live_test(drive_service,
roles_folder_id='YOUR-ROLES-FOLDER-ID-HERE',
              roles_doc_name='roles_1.json',
              docs_folder_id='YOUR-DOCS-FOLDER-ID-HERE',
              doc_filename='cfpb_supervisory-highlights-special-ed-auto-finance_2024-10.pdf',
              temp=0.3)

✅ Document text extracted.
📝 Summaries:

**Role Title:** ML Modeler for Auto Lending  
**Relevance:** Very Relevant  
**Summary:**  
This document is highly relevant and should be read in full. It provides critical insights into the auto finance market, highlighting significant supervisory observations and violations that impact model development for subprime auto lending. Key areas include deceptive marketing practices, origination disclosures, repossession activities, and issues related to add-on products. Understanding these regulatory findings will help in developing models that mitigate risks associated with high default rates and ensure compliance with consumer protection laws, particularly for underserved markets.

---

**Role Title:** SVP for Risk Analytics - Consumer Lending  
**Relevance:** Very Relevant  
**Summary:**  
This document is highly relevant and should be read in full. It outlines the Consumer Financial Protection Bureau's (CFPB) supervisory highlights regarding t